# Experiment 7.3.9 — Pretrained A2 depth extension

Analysis-only notebook for finalized Exp7.3.9 artifacts. Training and probe fitting are performed by the experiment script/Slurm pipeline.


In [1]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
ART = ROOT / 'notebooks' / 'artifacts' / 'experiment_7_3_9_pretrained_a2_depth_extension' / 'pretrained_a2_depth_extension_v1'
ART


PosixPath('/home/zhaolongwei_umass_edu/projects/writingRing/notebooks/artifacts/experiment_7_3_9_pretrained_a2_depth_extension/pretrained_a2_depth_extension_v1')

In [2]:
manifest = json.loads((ART / 'manifest.json').read_text())
native_runs = pd.read_csv(ART / 'native_runs.csv')
native_summary = pd.read_csv(ART / 'native_summary.csv')
probe_runs = pd.read_csv(ART / 'probe_runs.csv')
probe_summary = pd.read_csv(ART / 'probe_summary.csv')
depth_contrasts = pd.read_csv(ART / 'depth_contrasts.csv')
bias_contrasts = pd.read_csv(ART / 'bias_contrasts.csv')
temporal_contrasts = pd.read_csv(ART / 'temporal_contrasts.csv')
layer_contrasts = pd.read_csv(ART / 'layer_contrasts.csv')
manifest


{'architecture_3layer': {'output': 'bias-free Linear 128->12',
  'shifts': [[2, 3, 4], [2, 3, 4], [2, 3, 4]],
  'widths': [128, 128, 128]},
 'cases': {'C0_a2_2layer': 'reuse Exp7.3 A2 checkpoint; no retraining',
  'C1_frozen_a2_train_l3': 'A2 W1/W2/Wout frozen; insert 128-neuron (2,3,4) L3 and train W3 only',
  'C2_c1_init_e2e_3layer': 'load selected C1 checkpoint; fresh Adam; epoch 0 is C1 candidate; unfreeze W1/W2/W3/Wout'},
 'counts': {'native_rows': 9,
  'new_training_runs': 6,
  'probe_rows': 96,
  'probe_tasks': 9},
 'experiment_id': 'experiment_7_3_9_pretrained_a2_depth_extension',
 'files': {'bias_contrasts': 'bias_contrasts.csv',
  'depth_contrasts': 'depth_contrasts.csv',
  'layer_contrasts': 'layer_contrasts.csv',
  'native_runs': 'native_runs.csv',
  'native_summary': 'native_summary.csv',
  'probe_runs': 'probe_runs.csv',
  'probe_summary': 'probe_summary.csv',
  'temporal_contrasts': 'temporal_contrasts.csv'},
 'probes': {'C_grid': [0.001, 0.01, 0.1, 1.0, 10.0],
  'affine

## Native C0/C1/C2 performance


In [3]:
display(native_runs.sort_values(['case', 'seed']))
display(native_summary)
display(depth_contrasts)


,case,seed,architecture,training_scope,best_epoch,train_ba,val_ba,test_ba,test_accuracy,test_macro_f1
0,C0_a2_2layer,11,234x234,reused_exp7.3_A2,94,0.931670,0.523653,0.530632,0.547945,0.518542
1,C0_a2_2layer,23,234x234,reused_exp7.3_A2,89,0.917433,0.572264,0.596182,0.602740,0.589045
2,C0_a2_2layer,37,234x234,reused_exp7.3_A2,90,0.917487,0.566576,0.566691,0.575342,0.556651
3,C1_frozen_a2_train_l3,11,234x234x234,W3_only,9,0.866744,0.544060,0.479505,0.500000,0.459914
4,C1_frozen_a2_train_l3,23,234x234x234,W3_only,47,0.945803,0.563606,0.536384,0.541096,0.521352
5,C1_frozen_a2_train_l3,37,234x234x234,W3_only,10,0.836193,0.516679,0.410526,0.424658,0.391027
6,C2_c1_init_e2e_3layer,11,234x234x234,full_e2e_from_C1,0,0.866744,0.544060,0.479505,0.500000,0.459914
7,C2_c1_init_e2e_3layer,23,234x234x234,full_e2e_from_C1,18,0.977412,0.570881,0.530893,0.541096,0.522864
8,C2_c1_init_e2e_3layer,37,234x234x234,full_e2e_from_C1,58,0.987380,0.580267,0.502162,0.513699,0.485211


,case,architecture,training_scope,test_ba_mean,test_ba_std,test_ba_count,test_accuracy_mean,test_accuracy_std,test_accuracy_count,test_macro_f1_mean,test_macro_f1_std,test_macro_f1_count,best_epoch_mean,best_epoch_std,best_epoch_count
0,C0_a2_2layer,234x234,reused_exp7.3_A2,0.564502,0.032830,3,0.575342,0.027397,3,0.554746,0.035290,3,91.000000,2.645751,3
1,C1_frozen_a2_train_l3,234x234x234,W3_only,0.475472,0.063026,3,0.488584,0.059053,3,0.457431,0.065198,3,22.000000,21.656408,3
2,C2_c1_init_e2e_3layer,234x234x234,full_e2e_from_C1,0.504187,0.025754,3,0.518265,0.020925,3,0.489330,0.031676,3,25.333333,29.687259,3


,contrast,seed,left_case,right_case,test_ba_delta_pp
0,C1_minus_C0_insert_L3,11,C1_frozen_a2_train_l3,C0_a2_2layer,-5.112734
1,C1_minus_C0_insert_L3,23,C1_frozen_a2_train_l3,C0_a2_2layer,-5.979853
2,C1_minus_C0_insert_L3,37,C1_frozen_a2_train_l3,C0_a2_2layer,-15.616467
3,C2_minus_C1_e2e_adaptation,11,C2_c1_init_e2e_3layer,C1_frozen_a2_train_l3,0.000000
4,C2_minus_C1_e2e_adaptation,23,C2_c1_init_e2e_3layer,C1_frozen_a2_train_l3,-0.549034
5,C2_minus_C1_e2e_adaptation,37,C2_c1_init_e2e_3layer,C1_frozen_a2_train_l3,9.163545
6,C2_minus_C0_total_depth_gain,11,C2_c1_init_e2e_3layer,C0_a2_2layer,-5.112734
7,C2_minus_C0_total_depth_gain,23,C2_c1_init_e2e_3layer,C0_a2_2layer,-6.528888
8,C2_minus_C0_total_depth_gain,37,C2_c1_init_e2e_3layer,C0_a2_2layer,-6.452922


## Layer-wise representation probes


In [4]:
probe_view = probe_summary.sort_values(['case', 'layer', 'representation', 'bias_mode'])
display(probe_view)
display(layer_contrasts)


,case,layer,representation,bias_mode,test_balanced_accuracy_mean,test_balanced_accuracy_std,test_balanced_accuracy_count,test_accuracy_mean,test_accuracy_std,test_accuracy_count,test_macro_f1_mean,test_macro_f1_std,test_macro_f1_count,selected_C_mean,selected_C_std,selected_C_count
2,C0_a2_2layer,L1,fixed250,affine,0.544459,0.026306,3,0.557078,0.028516,3,0.545146,0.026375,3,0.700,0.519615,3
3,C0_a2_2layer,L1,fixed250,no_bias,0.551548,0.020307,3,0.563927,0.022017,3,0.546046,0.023109,3,0.010,0.000000,3
0,C0_a2_2layer,L1,wholecount,affine,0.600300,0.024233,3,0.609589,0.018122,3,0.590511,0.019052,3,0.700,0.519615,3
1,C0_a2_2layer,L1,wholecount,no_bias,0.564839,0.039974,3,0.573059,0.034474,3,0.556422,0.041466,3,0.700,0.519615,3
6,C0_a2_2layer,L2,fixed250,affine,0.579424,0.023212,3,0.593607,0.020925,3,0.578305,0.028892,3,0.370,0.547449,3
7,C0_a2_2layer,L2,fixed250,no_bias,0.580545,0.016500,3,0.595890,0.018122,3,0.575498,0.021290,3,3.370,5.741925,3
4,C0_a2_2layer,L2,wholecount,affine,0.586273,0.009744,3,0.600457,0.003954,3,0.581724,0.014811,3,0.010,0.000000,3
5,C0_a2_2layer,L2,wholecount,no_bias,0.540981,0.036907,3,0.547945,0.038135,3,0.532058,0.034654,3,6.670,5.767729,3
10,C1_frozen_a2_train_l3,L1,fixed250,affine,0.544459,0.026306,3,0.557078,0.028516,3,0.545146,0.026375,3,0.700,0.519615,3
11,C1_frozen_a2_train_l3,L1,fixed250,no_bias,0.551548,0.020307,3,0.563927,0.022017,3,0.546046,0.023109,3,0.010,0.000000,3


,case,seed,representation,bias_mode,L1,L2,L3,L3_minus_L2_pp
0,C1_frozen_a2_train_l3,11,fixed250,affine,0.523441,0.571629,0.551560,-2.006882
1,C1_frozen_a2_train_l3,11,fixed250,no_bias,0.535630,0.580235,0.556732,-2.350289
2,C1_frozen_a2_train_l3,11,wholecount,affine,0.577205,0.575830,0.514533,-6.129634
3,C1_frozen_a2_train_l3,11,wholecount,no_bias,0.530462,0.535766,0.502135,-3.363095
4,C1_frozen_a2_train_l3,23,fixed250,affine,0.535977,0.605531,0.539658,-6.587302
5,C1_frozen_a2_train_l3,23,fixed250,no_bias,0.544597,0.597197,0.534923,-6.227453
6,C1_frozen_a2_train_l3,23,wholecount,affine,0.598166,0.587870,0.562872,-2.499792
7,C1_frozen_a2_train_l3,23,wholecount,no_bias,0.555352,0.506960,0.544717,3.775738
8,C1_frozen_a2_train_l3,37,fixed250,affine,0.573960,0.561113,0.494061,-6.705239
9,C1_frozen_a2_train_l3,37,fixed250,no_bias,0.574418,0.564202,0.541182,-2.302004


## Bias and temporal-accessibility contrasts


In [5]:
display(bias_contrasts.sort_values(['case', 'seed', 'layer', 'representation']))
display(temporal_contrasts.sort_values(['case', 'seed', 'layer', 'bias_mode']))


,case,seed,layer,representation,affine,no_bias,affine_minus_no_bias_pp
0,C0_a2_2layer,11,L1,fixed250,0.523441,0.535630,-1.218920
1,C0_a2_2layer,11,L1,wholecount,0.577205,0.530462,4.674284
2,C0_a2_2layer,11,L2,fixed250,0.571629,0.580235,-0.860598
3,C0_a2_2layer,11,L2,wholecount,0.575830,0.535766,4.006410
4,C0_a2_2layer,23,L1,fixed250,0.535977,0.544597,-0.861985
5,C0_a2_2layer,23,L1,wholecount,0.598166,0.555352,4.281482
6,C0_a2_2layer,23,L2,fixed250,0.605531,0.597197,0.833333
7,C0_a2_2layer,23,L2,wholecount,0.587870,0.506960,8.091006
8,C0_a2_2layer,37,L1,fixed250,0.573960,0.574418,-0.045788
9,C0_a2_2layer,37,L1,wholecount,0.625529,0.608702,1.682692


,case,seed,layer,bias_mode,fixed250,wholecount,fixed250_minus_wholecount_pp
0,C0_a2_2layer,11,L1,affine,0.523441,0.577205,-5.376360
1,C0_a2_2layer,11,L1,no_bias,0.535630,0.530462,0.516844
2,C0_a2_2layer,11,L2,affine,0.571629,0.575830,-0.420066
3,C0_a2_2layer,11,L2,no_bias,0.580235,0.535766,4.446942
4,C0_a2_2layer,23,L1,affine,0.535977,0.598166,-6.218920
5,C0_a2_2layer,23,L1,no_bias,0.544597,0.555352,-1.075452
6,C0_a2_2layer,23,L2,affine,0.605531,0.587870,1.766081
7,C0_a2_2layer,23,L2,no_bias,0.597197,0.506960,9.023754
8,C0_a2_2layer,37,L1,affine,0.573960,0.625529,-5.156926
9,C0_a2_2layer,37,L1,no_bias,0.574418,0.608702,-3.428447


Interpretation targets:

- `C1 - C0`: value of an inserted L3 with A2 frozen.
- `C2 - C1`: gain/loss from coordinated E2E adaptation.
- `L3 - L2`: whether the new layer makes class information more directly accessible.
- `affine - no_bias`: dependence on class-specific constant offsets.
- `Fixed250 - WholeCount`: remaining dependence on explicit temporal position.
